# 00 · Build data (CPU, or Colab GPU for a local teacher)

Prose corpus → non-IID clients (prose shards + log slices + benign) → 4 task datasets → train/val/test splits.

**Set up first:** copy `.env.example`→`.env`. For matched both-class log data with no Splunk lab, download a paper-backed dataset (e.g. AIT), convert it below, and point `FEDDAPT_LOG_SOURCES` at the output.

In [3]:
import os, sys
if not os.path.exists('/content/fedapt/pyproject.toml'):
    !git clone https://github.com/dsuyu1/fedapt.git /content/fedapt
os.chdir('/content/fedapt')
!pip install -e ".[eval]"
sys.path.insert(0, os.path.abspath('src'))   # expose the src-layout pkg to this kernel
import fedapt; print('fedapt OK:', fedapt.__file__)

Obtaining file:///content/fedapt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.8 MB/s eta 0:00:00
  Building editable for fedapt (pyproject.toml) ... done
  Created wheel for fedapt: filename=fedapt-0.1.0-0.editable-py3-none-any.whl size=2336 sha256=b6700e1305779c492645d3117504e2b9f5dc4e8330c93e07c8370e4ddd397ff5
  Stored in directory: /tmp/pip-ephem-wheel-cache-2fv7q5rq/wheels/bf/53/c3/4dc2e7b687f07244f6f1c8bad3be6806a5e7dc97dad923ea44
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=c5d64fdbfd20b3c81eea03c791bf3984c461a

fedapt OK: None


### Optional: run a local teacher on Colab's GPU (no API key)
Installs Ollama + pulls a model; then select it as the teacher below. Skip for an API teacher or offline fallbacks.

In [ ]:
# --- OPTIONAL: run a strong LOCAL teacher/judge on Colab's GPU via Ollama ---
# Open-weight models (Qwen/Gemma/Llama) run on the SAME GPU as training, so only
# do this during data-build (nb 00) or the judge pass (nb 03), never during training.
# T4: qwen3:14b / gemma3:12b.  A100: qwen3:32b / gemma3:27b.
get_ipython().system('curl -fsSL https://ollama.com/install.sh | sh')
import subprocess, time, os
subprocess.Popen(['ollama', 'serve']); time.sleep(5)
get_ipython().system('ollama pull qwen3:14b')
os.environ['FEDDAPT_OLLAMA_HOST'] = 'http://localhost:11434'

In [4]:
from fedapt.config import load_config
from fedapt import corpus, clients, tasks, splits
cfg = load_config()
print('root =', cfg.root)

ModuleNotFoundError: No module named 'fedapt.config'

### Optional: convert a paper-backed dataset (both classes, matched)
Produces a normalized `.jsonl`; set `FEDDAPT_LOG_SOURCES=data/normalized` in `.env`, then rebuild.

In [ ]:
# after downloading AIT to Drive (both benign+attack, one matched environment):
# !python scripts/convert_dataset.py --dataset ait \
#     --src /content/drive/MyDrive/AIT/<dataset> --out data/normalized/ait.jsonl

### Teacher for real task targets (else `teacher=None` = metadata fallbacks)

In [ ]:
teacher = None
from fedapt.judge import make_llm
# LOCAL model on Colab's GPU (run the Ollama cell first):
# teacher = make_llm('ollama:qwen3:14b')
# or an API model (needs ANTHROPIC_API_KEY in .env and pip install -e '.[eval]'):
# teacher = make_llm('claude-haiku-4-5')

In [ ]:
corpus.build_corpus(cfg)
clients.build_clients(cfg)
tasks.build_tasks(cfg, teacher=teacher)   # watch the '⚠ fell back' line — it should be 0
splits.build_splits(cfg)

Next → **01 Federated DAPT** (GPU).